# MVTec AD — Exploratory Data Analysis

**目標：** 了解 MVTec AD 資料集結構、影像特性，以及正常品 vs 瑕疵品的差異。

**資料集：** [MVTec Anomaly Detection Dataset](https://www.mvtec.com/company/research/datasets/mvtec-ad)
- 15 個類別（物件 + 紋理）
- 訓練集：只有正常品
- 測試集：正常品 + 多種瑕疵類型
- Ground truth：pixel-level segmentation masks

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import defaultdict

DATA_ROOT = Path("../data")
CATEGORIES = sorted([d for d in os.listdir(DATA_ROOT) 
                     if (DATA_ROOT / d).is_dir() and d != "mvtec_ad_evaluation"])
print(f"Categories ({len(CATEGORIES)}): {CATEGORIES}")

## 1. 資料集統計：每個類別的影像數量

In [ ]:
# Count images per category and split
stats = []
for cat in CATEGORIES:
    cat_dir = DATA_ROOT / cat
    # Train (only good)
    train_good = len(list((cat_dir / "train" / "good").glob("*.png")))
    # Test
    test_dir = cat_dir / "test"
    test_counts = {}
    for defect in sorted(os.listdir(test_dir)):
        if (test_dir / defect).is_dir():
            test_counts[defect] = len(list((test_dir / defect).glob("*.png")))
    
    total_test = sum(test_counts.values())
    n_defect_types = len([k for k in test_counts if k != "good"])
    stats.append({
        "category": cat,
        "train_normal": train_good,
        "test_total": total_test,
        "test_normal": test_counts.get("good", 0),
        "test_anomaly": total_test - test_counts.get("good", 0),
        "defect_types": n_defect_types,
        "defect_names": [k for k in test_counts if k != "good"],
    })

# Display as table
print(f"{'Category':<15} {'Train':>6} {'Test':>6} {'Normal':>7} {'Anomaly':>8} {'Defects':>8}")
print("-" * 60)
for s in stats:
    print(f"{s['category']:<15} {s['train_normal']:>6} {s['test_total']:>6} "
          f"{s['test_normal']:>7} {s['test_anomaly']:>8} {s['defect_types']:>8}")
print("-" * 60)
total_train = sum(s["train_normal"] for s in stats)
total_test = sum(s["test_total"] for s in stats)
print(f"{'TOTAL':<15} {total_train:>6} {total_test:>6}")

## 2. 視覺化：正常品 vs 瑕疵品 + Ground Truth Mask

聚焦 3 個目標類別：**bottle, carpet, hazelnut**

In [ ]:
TARGET_CATS = ["bottle", "carpet", "hazelnut"]

for cat in TARGET_CATS:
    cat_dir = DATA_ROOT / cat
    defect_types = [d for d in sorted(os.listdir(cat_dir / "test")) 
                    if d != "good" and (cat_dir / "test" / d).is_dir()]
    
    n_cols = 1 + len(defect_types)  # 1 normal + N defect types
    fig, axes = plt.subplots(2, n_cols, figsize=(4 * n_cols, 8))
    fig.suptitle(f"Category: {cat}", fontsize=16, fontweight="bold")
    
    # Normal sample
    normal_img_path = sorted((cat_dir / "train" / "good").glob("*.png"))[0]
    normal_img = cv2.cvtColor(cv2.imread(str(normal_img_path)), cv2.COLOR_BGR2RGB)
    axes[0, 0].imshow(normal_img)
    axes[0, 0].set_title("Normal (train)", fontsize=11)
    axes[0, 0].axis("off")
    axes[1, 0].imshow(np.zeros_like(normal_img), cmap="gray")
    axes[1, 0].set_title("No mask", fontsize=11)
    axes[1, 0].axis("off")
    
    # Defect samples
    for i, defect in enumerate(defect_types):
        defect_dir = cat_dir / "test" / defect
        gt_dir = cat_dir / "ground_truth" / defect
        
        img_path = sorted(defect_dir.glob("*.png"))[0]
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        
        mask_name = img_path.name.replace(".png", "_mask.png")
        mask_path = gt_dir / mask_name
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        
        axes[0, i + 1].imshow(img)
        axes[0, i + 1].set_title(f"Defect: {defect}", fontsize=11)
        axes[0, i + 1].axis("off")
        
        axes[1, i + 1].imshow(mask, cmap="hot")
        axes[1, i + 1].set_title("GT Mask", fontsize=11)
        axes[1, i + 1].axis("off")
    
    plt.tight_layout()
    plt.show()

## 3. OpenCV 前處理效果展示

展示 OpenCV pipeline 對影像的處理效果：灰階化、高斯模糊、邊緣偵測、形態學操作。

In [ ]:
# OpenCV preprocessing showcase on a defective bottle image
cat_dir = DATA_ROOT / "bottle"
defect_img_path = sorted((cat_dir / "test" / "broken_large").glob("*.png"))[0]
img_bgr = cv2.imread(str(defect_img_path))
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle("OpenCV Preprocessing Pipeline (bottle — broken_large)", fontsize=14, fontweight="bold")

# Row 1: Basic transforms
axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title("Original")

gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
axes[0, 1].imshow(gray, cmap="gray")
axes[0, 1].set_title("Grayscale")

blurred = cv2.GaussianBlur(gray, (5, 5), 0)
axes[0, 2].imshow(blurred, cmap="gray")
axes[0, 2].set_title("Gaussian Blur (5x5)")

edges = cv2.Canny(blurred, 50, 150)
axes[0, 3].imshow(edges, cmap="gray")
axes[0, 3].set_title("Canny Edge Detection")

# Row 2: Thresholding + morphology
_, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
axes[1, 0].imshow(thresh, cmap="gray")
axes[1, 0].set_title("Otsu Thresholding")

adaptive = cv2.adaptiveThreshold(blurred, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                  cv2.THRESH_BINARY, 11, 2)
axes[1, 1].imshow(adaptive, cmap="gray")
axes[1, 1].set_title("Adaptive Threshold")

kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
morph = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
axes[1, 2].imshow(morph, cmap="gray")
axes[1, 2].set_title("Morphological Close")

# CLAHE (Contrast Limited Adaptive Histogram Equalization)
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
enhanced = clahe.apply(gray)
axes[1, 3].imshow(enhanced, cmap="gray")
axes[1, 3].set_title("CLAHE Enhancement")

for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 4. 影像統計：像素值分布（正常 vs 瑕疵）

比較正常品和瑕疵品的像素值分布，觀察是否有明顯差異。

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, cat in enumerate(TARGET_CATS):
    cat_dir = DATA_ROOT / cat
    
    # Sample normal images
    normal_paths = sorted((cat_dir / "train" / "good").glob("*.png"))[:30]
    normal_means = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).mean() for p in normal_paths]
    normal_stds = [cv2.imread(str(p), cv2.IMREAD_GRAYSCALE).std() for p in normal_paths]
    
    # Sample anomaly images
    anomaly_means, anomaly_stds = [], []
    for defect in os.listdir(cat_dir / "test"):
        if defect == "good" or not (cat_dir / "test" / defect).is_dir():
            continue
        for p in sorted((cat_dir / "test" / defect).glob("*.png"))[:10]:
            img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
            anomaly_means.append(img.mean())
            anomaly_stds.append(img.std())
    
    axes[i].scatter(normal_means, normal_stds, alpha=0.6, label="Normal", c="green", s=40)
    axes[i].scatter(anomaly_means, anomaly_stds, alpha=0.6, label="Anomaly", c="red", s=40)
    axes[i].set_xlabel("Mean Pixel Value")
    axes[i].set_ylabel("Std Pixel Value")
    axes[i].set_title(f"{cat}")
    axes[i].legend()

fig.suptitle("Pixel Statistics: Normal vs Anomaly", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nKey insight: 像素值統計分布往往重疊很大 → 傳統閾值法不夠用 → 需要深度學習學習更高階的特徵表達")

## 5. 瑕疵區域大小分析

從 ground truth mask 計算瑕疵佔整張影像的比例 — 了解偵測的難度。

In [ ]:
defect_ratios = defaultdict(list)

for cat in TARGET_CATS:
    gt_dir = DATA_ROOT / cat / "ground_truth"
    for defect in sorted(os.listdir(gt_dir)):
        if not (gt_dir / defect).is_dir():
            continue
        for mask_path in (gt_dir / defect).glob("*.png"):
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            ratio = (mask > 127).sum() / mask.size * 100
            defect_ratios[cat].append(ratio)

fig, ax = plt.subplots(figsize=(10, 5))
positions = range(len(TARGET_CATS))
bp = ax.boxplot([defect_ratios[cat] for cat in TARGET_CATS], 
                positions=positions, widths=0.5, patch_artist=True)

colors = ["#3498db", "#e74c3c", "#2ecc71"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)

ax.set_xticks(positions)
ax.set_xticklabels(TARGET_CATS)
ax.set_ylabel("Defect Area (%)")
ax.set_title("Defect Region Size as % of Total Image Area", fontweight="bold")
plt.tight_layout()
plt.show()

for cat in TARGET_CATS:
    ratios = defect_ratios[cat]
    print(f"{cat}: median={np.median(ratios):.1f}%, mean={np.mean(ratios):.1f}%, max={np.max(ratios):.1f}%")

## Next Steps

1. **`src/train.py`** — 用正常品訓練 ResNet18 Autoencoder
2. **`src/evaluate.py`** — 計算 Image-level & Pixel-level AUROC
3. **`src/visualize.py`** — 產生瑕疵熱力圖

```bash
# 訓練
cd ../src && python train.py --category bottle --epochs 100

# 評估
python evaluate.py --category bottle

# 視覺化
python visualize.py --category bottle
```